In [1]:
import os, pickle, re
import numpy as np
import pandas as pd
import json
import hist

from physics.simulation import mcfm, msq
from physics.hstar import eft
from physics.variables import eft_terms, c6_degree, ct_degree, cg_degree
from physics.constants import c6_to_cH, cg_to_cHG, ct_to_ctH, cH_to_c6
from nsbi import carl, taylr

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [2]:
torch.set_float32_matmul_precision('medium')

import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": r"\usepackage{amssymb}",
})

In [3]:
run_dir_4l = 'run/presel/h4l'
run_dir_2l2v = 'run/presel/h2l2v'

In [4]:
(events_gg_4l_train, events_gg_4l), scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l = taylr.utils.load_results(run_dir_4l, eft_terms)
(events_gg_2l2v_train, events_gg_2l2v), scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v = taylr.utils.load_results(run_dir_2l2v, eft_terms)

In [5]:
(events_qq_4l_train, events_qq_4l), _, scaler_carl_4l, model_carl_4l = carl.utils.load_results(run_dir_4l)
(events_qq_2l2v_train, events_qq_2l2v), _, scaler_carl_2l2v, model_carl_2l2v = carl.utils.load_results(run_dir_2l2v)

In [6]:
lumi = 3000.

c6_linspace = [-25,25,51]
c6_space = np.linspace(*c6_linspace)
c6_sm = 0.0
i_c6_sm = np.where(np.isclose(c6_space,c6_sm))[0][0]
# c6_asimov = -1.0
c6_asimov = 100
cH_asimov = c6_asimov * c6_to_cH

In [7]:
class ZeroWeightFilter():
    def __init__(self):
        pass
    def __call__(self, kinematics, components, weights, probabilities):
        return np.where(weights != 0)

sample_size = 1_000_000
random_state = 42

# CALIBRATION

# events_gg_4l_calib, events_gg_2l2v_calib = events_gg_4l_calib.sample(sample_size,random_state=random_state), events_gg_2l2v_calib.filter(ZeroWeightFilter()).sample(sample_size,random_state=random_state)
# events_qq_4l_calib, events_qq_2l2v_calib = events_qq_4l_calib.sample(sample_size,random_state=random_state), events_qq_2l2v_calib.filter(ZeroWeightFilter()).sample(sample_size,random_state=random_state)

# events_4l_calib = mcfm.stack(events_gg_4l_calib, events_qq_4l_calib)
# events_2l2v_calib = mcfm.stack(events_gg_2l2v_calib, events_qq_2l2v_calib)

# TESTING

events_gg_4l, events_gg_2l2v = events_gg_4l.sample(sample_size,random_state=random_state), events_gg_2l2v.filter(ZeroWeightFilter()).sample(sample_size,random_state=random_state)
events_qq_4l, events_qq_2l2v = events_qq_4l.sample(sample_size,random_state=random_state), events_qq_2l2v.filter(ZeroWeightFilter()).sample(sample_size,random_state=random_state)

w_gg_4l_sm, w_gg_2l2v_sm = events_gg_4l.weights.to_numpy(), events_gg_2l2v.weights.to_numpy()
w_qq_4l_sm, w_qq_2l2v_sm = events_qq_4l.weights.to_numpy(), events_qq_2l2v.weights.to_numpy()

sigma_gg_4l_sm = np.sum(w_gg_4l_sm)
sigma_gg_2l2v_sm = np.sum(w_gg_2l2v_sm)
sigma_qq_4l_sm = np.sum(w_qq_4l_sm)
sigma_qq_2l2v_sm = np.sum(w_qq_2l2v_sm)

events_4l = mcfm.stack(events_gg_4l, events_qq_4l)
events_2l2v = mcfm.stack(events_gg_2l2v, events_qq_2l2v)
sigma_4l_sm = events_4l.weights.sum()
sigma_2l2v_sm = events_2l2v.weights.sum()

events_gg = mcfm.stack(events_gg_4l, events_gg_2l2v)
events_qq = mcfm.stack(events_qq_4l, events_qq_2l2v)
sigma_gg_sm = events_gg.weights.sum()
sigma_qq = events_qq.weights.sum()

In [8]:
features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
features_2l2v = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

In [9]:
# 4l

w_gg_4l_sm = torch.tensor(events_gg_4l.weights.to_numpy())
sigma_gg_4l_sm = torch.sum(w_gg_4l_sm)

eft_mod_gg_4l = eft.Modifier(events=events_gg_4l)

w_gg_4l_c6, _ =  torch.tensor(eft_mod_gg_4l.modify(torch.tensor(c6_space), None, None))
sigma_gg_4l_c6 = torch.sum(w_gg_4l_c6, dim=0)  # c6 & cHbox

# 2l2v

w_gg_2l2v_sm = torch.tensor(events_gg_2l2v.weights.to_numpy())
sigma_gg_2l2v_sm = torch.sum(w_gg_2l2v_sm)

eft_mod_gg_2l2v = eft.Modifier(events=events_gg_2l2v)

w_gg_2l2v_c6, _ =  torch.tensor(eft_mod_gg_2l2v.modify(torch.tensor(c6_space), None, None))
sigma_gg_2l2v_c6 = torch.sum(w_gg_2l2v_c6, dim=0)  # c6 & cHbox

/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(val, dtype=dtype), np.isscalar(val)
/tmp/ipykernel_3081067/220552909.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1708025847130/work/torch/csrc/utils/tensor_new.cpp:275.)
  w_gg_4l_c6, _ =  torch.tensor(eft_mod_gg_4l.modify(torch.tensor(c6_space), None, None))
/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torc

In [10]:
# # TEST: getting the ordering of events wrong should mess with everything below
# # w_pp_sm = mcfm.stack(events_gg_2l2v,events_qq_4l,events_gg_4l,events_qq_2l2v).weights.to_numpy()

w_pp_4l_sm = torch.tensor(events_4l.weights.to_numpy())
sigma_pp_4l_sm = torch.sum(w_pp_4l_sm)
p_pp_4l_sm = w_pp_4l_sm / sigma_pp_4l_sm
nu_pp_4l_sm = sigma_pp_4l_sm * lumi

sigma_pp_4l_c6 = sigma_gg_4l_c6 + sigma_qq_4l_sm
nu_pp_4l_c6 = sigma_pp_4l_c6 * lumi

w_pp_2l2v_sm = torch.tensor(events_2l2v.weights.to_numpy())
sigma_pp_2l2v_sm = torch.sum(w_pp_2l2v_sm)
p_pp_2l2v_sm = w_pp_2l2v_sm / sigma_pp_2l2v_sm
nu_pp_2l2v_sm = sigma_pp_2l2v_sm * lumi

sigma_pp_2l2v_c6 = sigma_gg_2l2v_c6 + sigma_qq_2l2v_sm
nu_pp_2l2v_c6 = sigma_pp_2l2v_c6 * lumi

In [11]:
# 4l

eft_mod_gg_4l = eft.Modifier(events=events_gg_4l)

w_gg_4l_c6_asimov, _ = eft_mod_gg_4l.modify(c6_asimov, None, None)
w_gg_4l_c6_asimov = torch.tensor(w_gg_4l_c6_asimov)
sigma_gg_4l_c6_asimov = torch.sum(w_gg_4l_c6_asimov, dim=0)

w_pp_4l_c6_asimov = torch.cat((w_gg_4l_c6_asimov, torch.tensor(events_qq_4l.weights.to_numpy())), dim=0)
sigma_pp_4l_c6_asimov = torch.sum(w_pp_4l_c6_asimov, dim=0)
nu_pp_4l_c6_asimov = sigma_pp_4l_c6_asimov * lumi

# 2l2v

eft_mod_gg_2l2v = eft.Modifier(events=events_gg_2l2v)

w_gg_2l2v_c6_asimov, _ = eft_mod_gg_2l2v.modify(c6_asimov, None, None)
w_gg_2l2v_c6_asimov = torch.tensor(w_gg_2l2v_c6_asimov)
sigma_gg_2l2v_c6_asimov = torch.sum(w_gg_2l2v_c6_asimov, dim=0)

w_pp_2l2v_c6_asimov = torch.cat((w_gg_2l2v_c6_asimov, torch.tensor(events_qq_2l2v.weights.to_numpy())), dim=0)
sigma_pp_2l2v_c6_asimov = torch.sum(w_pp_2l2v_c6_asimov, dim=0)
nu_pp_2l2v_c6_asimov = sigma_pp_2l2v_c6_asimov * lumi

In [12]:
r_4l = carl.utils.get_likelihood_ratio(events_4l, features_4l, scaler_carl_4l, model_carl_4l)
r_2l2v = carl.utils.get_likelihood_ratio(events_2l2v, features_2l2v, scaler_carl_2l2v, model_carl_2l2v)
r = torch.cat((r_4l, r_2l2v))

coeffs_4l = taylr.utils.get_coefficients(events_4l, features_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l)
coeffs_2l2v = taylr.utils.get_coefficients(events_2l2v, features_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v)
coeffs = torch.cat((coeffs_4l,coeffs_2l2v),dim=0)

coeffs_gg_4l = taylr.utils.get_coefficients(events_gg_4l, features_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l)
coeffs_gg_2l2v = taylr.utils.get_coefficients(events_gg_2l2v, features_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v)
coeffs_gg = torch.cat((coeffs_gg_4l,coeffs_gg_2l2v),dim=0)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425:

In [13]:
# 4l

carl_4l = sigma_qq_4l_sm * torch.tensor(r_4l)[:,None]
taylr_4l_c6 = sigma_gg_4l_sm * torch.tensor(eft.msq_eft_over_sm(coeffs_4l, [c6_asimov], None, None))
pratio_pp_4l_c6 =  (sigma_gg_4l_sm + sigma_qq_4l_sm)/(sigma_gg_4l_c6_asimov+sigma_qq_4l_sm) * (taylr_4l_c6 + carl_4l)/(sigma_gg_4l_sm + carl_4l)

# 2l2v

carl_2l2v = sigma_qq_2l2v_sm * torch.tensor(r_2l2v)[:,None]
taylr_2l2v_c6 = sigma_gg_2l2v_sm * torch.tensor(eft.msq_eft_over_sm(coeffs_2l2v, [c6_asimov], None, None))
pratio_pp_2l2v_c6 =  (sigma_gg_2l2v_sm + sigma_qq_2l2v_sm)/(sigma_gg_2l2v_c6_asimov+sigma_qq_2l2v_sm) * (taylr_2l2v_c6 + carl_2l2v)/(sigma_gg_2l2v_sm + carl_2l2v)

/tmp/ipykernel_3081067/3239430792.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  carl_4l = sigma_qq_4l_sm * torch.tensor(r_4l)[:,None]
/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eft_coefficients = torch.tensor(eft_coefficients)


/tmp/ipykernel_3081067/3239430792.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  carl_2l2v = sigma_qq_2l2v_sm * torch.tensor(r_2l2v)[:,None]


In [30]:
# r_hat = pratio_pp_4l_c6[:,0]
# s_hat = r_hat / (1+r_hat)
# p_sm = w_pp_4l_sm / sigma_pp_4l_sm
# p_bsm = w_pp_4l_c6_asimov / sigma_pp_4l_c6_asimov

r_hat = pratio_pp_2l2v_c6[:,0]
s_hat = r_hat / (1+r_hat)
p_sm = w_pp_2l2v_sm / sigma_pp_2l2v_sm
p_bsm = w_pp_2l2v_c6_asimov / sigma_pp_2l2v_c6_asimov

In [31]:
s_bins = np.linspace(0.0, 1.0, 41)
s_centers = (s_bins[:-1] + s_bins[1:])/2
s_nbins = len(s_bins) -1
s_widths = (s_bins[1:] - s_bins[:-1])/2

In [32]:

s_hat_binned = [s_hat[(s_hat >= s_bins[i]) & (s_hat < s_bins[i+1])] for i in range(s_nbins)]

p_sm_binned = [p_sm[(s_hat >= s_bins[i]) & (s_hat < s_bins[i+1])] for i in range(s_nbins)]
p_bsm_binned = [p_bsm[(s_hat >= s_bins[i]) & (s_hat < s_bins[i+1])] for i in range(s_nbins)]

sig_per_bin = np.array([np.sum(np.array(p_bsm_binned[i])) for i in range(s_nbins)])
bkg_per_bin = np.array([np.sum(np.array(p_sm_binned[i])) for i in range(s_nbins)])

s_true_binned = sig_per_bin/(sig_per_bin+bkg_per_bin)

sig_err_per_bin = np.sqrt(np.array([np.sum(np.array(p_bsm_binned[i])*np.array(p_bsm_binned[i]))  for i in range(s_nbins)]))
bkg_err_per_bin = np.sqrt(np.array([np.sum(np.array(p_sm_binned[i])*np.array(p_sm_binned[i]))  for i in range(s_nbins)]))
s_err_binned = np.sqrt(
    (sig_err_per_bin * bkg_per_bin / (sig_per_bin + bkg_per_bin)**2)**2 +
    (bkg_err_per_bin * sig_per_bin / (sig_per_bin + bkg_per_bin)**2)**2
)

/tmp/ipykernel_3081067/498197354.py:9: RuntimeWarning: invalid value encountered in divide
  s_true_binned = sig_per_bin/(sig_per_bin+bkg_per_bin)
/tmp/ipykernel_3081067/498197354.py:14: RuntimeWarning: invalid value encountered in divide
  (sig_err_per_bin * bkg_per_bin / (sig_per_bin + bkg_per_bin)**2)**2 +
/tmp/ipykernel_3081067/498197354.py:15: RuntimeWarning: invalid value encountered in divide
  (bkg_err_per_bin * sig_per_bin / (sig_per_bin + bkg_per_bin)**2)**2


In [34]:
lw= 1.5

fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [4,1]}, figsize=(5,5), sharex=True)
ax1.set_aspect('equal', adjustable='box')

ax1.errorbar([0,1], [0,1], color='grey', linestyle='--', linewidth=lw, label='$\\mathrm{MC}$')
ax1.errorbar(s_centers, s_true_binned, xerr=s_widths, yerr=s_err_binned, color='cornflowerblue', linestyle='none', linewidth=lw,  label='$\\mathrm{NSBI}$')

ax1.set_ylim(0,1)
ax1.set_ylabel('$\\mathrm{MC\\ estimate}\\ \\frac{p_{\\mathrm{BSM}}(x)}{ p_{\\mathrm{SM}}(x) + p_{\\mathrm{BSM}}(x) }$', fontsize=15)

ax1.legend(frameon=False, fontsize=12)

ax2.errorbar([0,1], [0,0], color='grey', linestyle='--', linewidth=lw)
ax2.errorbar(s_centers, np.array(s_true_binned)-np.array(s_centers), xerr=s_widths, yerr=s_err_binned, color='cornflowerblue', linestyle='none', linewidth=lw)

ax2.set_xlim(0,1)
ax2.set_xlabel('$\\mathrm{NSBI\\ estimate}\\ \\hat{s}(x)$', fontsize=15)
ax2.set_ylabel('$\\mathrm{Residual}$', fontsize=15)
ax2.set_ylim(-0.075, 0.075)
ax2.yaxis.set_ticks([-0.05, 0.05])

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)

ax1.text(0.96 ,0.20, f'$C_H = {c6_asimov * c6_to_cH:.1f}$', transform=ax1.transAxes, ha='right', va='bottom', fontsize=12)
# ax1.text(0.96 ,0.12, '$pp \\rightarrow ZZ \\rightarrow 4\\ell$', transform=ax1.transAxes, ha='right', va='bottom', fontsize=12)
ax1.text(0.96 ,0.12, '$pp \\rightarrow ZZ \\rightarrow 2\\ell 2\\nu$', transform=ax1.transAxes, ha='right', va='bottom', fontsize=12)
ax1.text(0.96 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV}$', transform=ax1.transAxes, ha='right', va='bottom', fontsize=12)

plt.tight_layout()
plt.subplots_adjust(hspace=0)

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

ax2.xaxis.set_tick_params(which='both', labeltop=False, top=True)

plt.savefig('s_calibration_2l2v.pdf', bbox_inches='tight')
plt.close()